In [1]:
import os
import numpy as np

import qkeras
from qkeras import QActivation
from qkeras import QConv1D
from qkeras.quantizers import quantized_bits, quantized_relu, quantized_tanh

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, UpSampling1D, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger



resolution = 150
filters = 4
name =f"no_opt_{resolution}_{filters}"
X_total = np.load(f"Data/X_Data_Bank_{resolution}.npy")
Y_total = np.load(f"Data/Y_Data_Bank_{resolution}.npy")
X_data = X_total[:1500,:,:]
y_data = Y_total[:1500,:,:]
TIME_STEPS = np.shape(X_data)[1]

VAL_SPLIT = 0.1

SAVE_DIR_opt = f'Output_drives/1Conv_checkpoints_running_{name}'
LOG_FILE_opt = f'Log_files/1Conv_3pulse_noise_tb23_{name}.csv'
MODEL_NAME_TEMPLATE_opt = '1Conv_2pulse_noise.loss_{val_loss:01.5f}.e{epoch:03d}_deconv_' + f'{name}.h5'
checkpoint_path_opt = os.path.join(SAVE_DIR_opt, MODEL_NAME_TEMPLATE_opt)


wq = quantized_bits(8, 2, alpha=1) 
aq = quantized_relu(6) 

qmodel = Sequential([
    Input(shape=(TIME_STEPS, 1)),

    QConv1D(filters, 3, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),
    
    QConv1D(filters, 3, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),

    QConv1D(filters, 3, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),
    
    QConv1D(1, 1, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    
    QActivation("quantized_sigmoid")
    ])

qmodel.summary()


optimizer = Adam(learning_rate=0.001)
qmodel.compile(loss='binary_crossentropy', optimizer=optimizer) 


if not os.path.isdir(SAVE_DIR_opt):
    os.makedirs(SAVE_DIR_opt)

callbacks = [
    ModelCheckpoint(checkpoint_path_opt, monitor="val_loss", save_best_only=True, mode="min", verbose=1),
    CSVLogger(LOG_FILE_opt, append=True, separator=','),
    EarlyStopping(monitor="val_loss", mode="min", patience=20, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=20, min_lr=1e-6, verbose=1) 
]


qmodel.fit(
    X_data, y_data,
    epochs=300,                  
    shuffle=True,
    validation_split=VAL_SPLIT,
    callbacks=callbacks
)

if not os.path.isdir("hgdream_new"):
    os.makedirs("hgdream_new")
    
qmodel.save(f"hgdream_new/quantized_model_{name}.h5")

2026-05-14 13:41:30.969403: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-14 13:41:31.007739: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-14 13:41:32.146345: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with 

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 q_conv1d (QConv1D)          (None, 80, 4)             16        
                                                                 
 q_activation (QActivation)  (None, 80, 4)             0         
                                                                 
 q_conv1d_1 (QConv1D)        (None, 80, 4)             52        
                                                                 
 q_activation_1 (QActivation  (None, 80, 4)            0         
 )                                                               
                                                                 
 q_conv1d_2 (QConv1D)        (None, 80, 4)           

Epoch 21/300
 1/43 [..............................] - ETA: 0s - loss: 0.1224
Epoch 21: val_loss improved from 0.14441 to 0.14327, saving model to Output_drives/1Conv_checkpoints_running_no_opt_150_4/1Conv_2pulse_noise.loss_0.14327.e021_deconv_no_opt_150_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 0.1464 - val_loss: 0.1433 - lr: 0.0010
Epoch 22/300
 1/43 [..............................] - ETA: 0s - loss: 0.1506
Epoch 22: val_loss did not improve from 0.14327
43/43 [==============================] - 0s 1ms/step - loss: 0.1464 - val_loss: 0.1450 - lr: 0.0010
Epoch 23/300
 1/43 [..............................] - ETA: 0s - loss: 0.1062
Epoch 23: val_loss improved from 0.14327 to 0.14228, saving model to Output_drives/1Conv_checkpoints_running_no_opt_150_4/1Conv_2pulse_noise.loss_0.14228.e023_deconv_no_opt_150_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 0.1463 - val_loss: 0.1423 - lr: 0.0010
Epoch 24/300
 1/43 [..............................] - ETA

 1/43 [..............................] - ETA: 0s - loss: 0.1290
Epoch 49: val_loss improved from 0.12626 to 0.12568, saving model to Output_drives/1Conv_checkpoints_running_no_opt_150_4/1Conv_2pulse_noise.loss_0.12568.e049_deconv_no_opt_150_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 0.1341 - val_loss: 0.1257 - lr: 0.0010
Epoch 50/300
 1/43 [..............................] - ETA: 0s - loss: 0.1192
Epoch 50: val_loss did not improve from 0.12568
43/43 [==============================] - 0s 1ms/step - loss: 0.1305 - val_loss: 0.1260 - lr: 0.0010
Epoch 51/300
 1/43 [..............................] - ETA: 0s - loss: 0.1262
Epoch 51: val_loss did not improve from 0.12568
43/43 [==============================] - 0s 1ms/step - loss: 0.1313 - val_loss: 0.1277 - lr: 0.0010
Epoch 52/300
 1/43 [..............................] - ETA: 0s - loss: 0.1255
Epoch 52: val_loss did not improve from 0.12568
43/43 [==============================] - 0s 1ms/step - loss: 0.1334 - val_loss:

In [2]:
import hls4ml
from qkeras.utils import _add_supported_quantized_objects

co = {}
_add_supported_quantized_objects(co)
print(co)
hls_config = hls4ml.utils.config_from_keras_model(
    qmodel,
    granularity='name',
    backend='Vitis'
)
hls_model = hls4ml.converters.convert_from_keras_model(
    qmodel,
    hls_config=hls_config,
    backend='Vitis',
    output_dir='model_final',
    part='xcu250-figd2104-2L-e',
    io_type='io_stream',
)
hls_model.compile()

Matplotlib created a temporary config/cache directory at /tmp/matplotlib-rxbp_6z9 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
/opt/conda/lib/python3.10/site-packages/hls4ml/converters/__init__.py:27: UserWarning: WARNING: Pytorch converter is not enabled!
  warnings.warn("WARNING: Pytorch converter is not enabled!", stacklevel=1)


{'QInitializer': <class 'qkeras.qlayers.QInitializer'>, 'QDense': <class 'qkeras.qlayers.QDense'>, 'QConv1D': <class 'qkeras.qconvolutional.QConv1D'>, 'QConv2D': <class 'qkeras.qconvolutional.QConv2D'>, 'QConv2DTranspose': <class 'qkeras.qconvolutional.QConv2DTranspose'>, 'QSimpleRNNCell': <class 'qkeras.qrecurrent.QSimpleRNNCell'>, 'QSimpleRNN': <class 'qkeras.qrecurrent.QSimpleRNN'>, 'QLSTMCell': <class 'qkeras.qrecurrent.QLSTMCell'>, 'QLSTM': <class 'qkeras.qrecurrent.QLSTM'>, 'QGRUCell': <class 'qkeras.qrecurrent.QGRUCell'>, 'QGRU': <class 'qkeras.qrecurrent.QGRU'>, 'QBidirectional': <class 'qkeras.qrecurrent.QBidirectional'>, 'QDepthwiseConv2D': <class 'qkeras.qconvolutional.QDepthwiseConv2D'>, 'QSeparableConv1D': <class 'qkeras.qconvolutional.QSeparableConv1D'>, 'QSeparableConv2D': <class 'qkeras.qconvolutional.QSeparableConv2D'>, 'QActivation': <class 'qkeras.qlayers.QActivation'>, 'QAdaptiveActivation': <class 'qkeras.qlayers.QAdaptiveActivation'>, 'QBatchNormalization': <class